In [ ]:
import os
import csv
import copy
import itertools
import cv2 as cv
import mediapipe as mp
import numpy as np

In [ ]:
def calc_landmark_list(image, landmarks):
    image_width, image_height = image.shape[1], image.shape[0]
    landmark_point = []

    for _, landmark in enumerate(landmarks.landmark):
        landmark_x = min(int(landmark.x * image_width), image_width - 1)
        landmark_y = min(int(landmark.y * image_height), image_height - 1)
        landmark_point.append([landmark_x, landmark_y])

    return landmark_point

In [ ]:
def pre_process_landmark(landmark_list):
    temp_landmark_list = copy.deepcopy(landmark_list)

    base_x, base_y = temp_landmark_list[0][0], temp_landmark_list[0][1]
    for index, _ in enumerate(temp_landmark_list):
        temp_landmark_list[index][0] = temp_landmark_list[index][0] - base_x
        temp_landmark_list[index][1] = temp_landmark_list[index][1] - base_y

    temp_landmark_list = list(itertools.chain.from_iterable(temp_landmark_list))
    max_value = max(list(map(abs, temp_landmark_list)))

    def normalize_(n):
        return n / max_value

    if max_value > 0:
        temp_landmark_list = list(map(normalize_, temp_landmark_list))

    return temp_landmark_list

In [ ]:
DATASET_DIR = "dataset/leapGestRecog"
OUT_CSV_PATH = "model/keypoint_classifier/keypoint.csv"
MIN_DETECTION_CONFIDENCE = 0.5

In [ ]:
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=MIN_DETECTION_CONFIDENCE,
)

success_count = 0
fail_count = 0

with open(OUT_CSV_PATH, 'w', newline="") as f:
    writer = csv.writer(f)
    
    if not os.path.exists(DATASET_DIR):
        print(f"Алдаа: {DATASET_DIR} хавтас олдсонгүй!")
    else:
        subject_folders = sorted(os.listdir(DATASET_DIR))
        for subject in subject_folders:
            subject_path = os.path.join(DATASET_DIR, subject)
            if not os.path.isdir(subject_path):
                continue

            gesture_folders = sorted(os.listdir(subject_path))
            for gesture in gesture_folders:
                gesture_path = os.path.join(subject_path, gesture)
                if not os.path.isdir(gesture_path):
                    continue

                try:
                    label_id = int(gesture.split('_')[0]) - 1
                except ValueError:
                    continue

                print(f"Боловсруулж байна: Subject {subject} -> Gesture {gesture}...")

                image_files = sorted(os.listdir(gesture_path))
                for img_file in image_files:
                    if not img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                        continue

                    img_path = os.path.join(gesture_path, img_file)
                    original_image = cv.imread(img_path)
                    if original_image is None:
                        continue

                    for flip_mode in [None, 1]: 
                        if flip_mode is None:
                            process_img = original_image
                        else:
                            process_img = cv.flip(original_image, flip_mode)

                        image_rgb = cv.cvtColor(process_img, cv.COLOR_BGR2RGB)
                        results = hands.process(image_rgb)

                        if results.multi_hand_landmarks is not None:
                            for hand_landmarks in results.multi_hand_landmarks:
                                landmark_list = calc_landmark_list(process_img, hand_landmarks)
                                pre_processed_landmark_list = pre_process_landmark(landmark_list)
                                writer.writerow([label_id, *pre_processed_landmark_list])
                                success_count += 1
                        else:
                            fail_count += 1

print("\n=========================================")
print(f"Ажиллагаа дууслаа! Файл: '{OUT_CSV_PATH}'")
print(f"Амжилттай танигдсан: {success_count} (Оригинал + Flip)")
print(f"Танигдаагүй: {fail_count}")
print("=========================================")